# 🧠 Engram — Persistent KV-Cache Inference Server

> Named, tiered, persistent KV-cache sessions for LLM inference.  
> Only the *delta* tokens are prefilled on each turn — not the full history.

## What this notebook does
1. Mounts Google Drive to cache the build binary (compile once, reuse forever)
2. Checks GPU availability
3. Clones and builds Engram with CUDA + ccache (fast on repeat runs)
4. Downloads a GGUF model from Hugging Face
5. Starts the Engram server in the background
6. Runs a **baseline** benchmark (full history re-prefilled every turn)
7. Runs an **Engram chat** benchmark (delta-only prefill)
8. Prints a side-by-side comparison

---
### ⚠️ Before running
- **Runtime → Change runtime type → T4 GPU**
- Run cells top-to-bottom
- **First build: ~25–40 min** (CUDA kernel compilation — Colab CPUs are slow)
- **Subsequent runs: ~10 seconds** (binary cached to your Google Drive)

## 0. Mount Google Drive (build cache)

In [ ]:
from google.colab import drive
drive.mount('/drive')

import os
CACHE_DIR = '/drive/MyDrive/engram_cache'
os.makedirs(CACHE_DIR, exist_ok=True)
print(f'✅ Drive mounted. Cache dir: {CACHE_DIR}')

## 1. Check GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
import subprocess
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
if r.returncode == 0:
    print(f'✅ GPU: {r.stdout.strip()}')
else:
    print('⚠️  No GPU — set Runtime → Change runtime type → GPU')

## 2. Install build tools + ccache

In [ ]:
%%bash
set -e
apt-get update -qq
apt-get install -y -qq cmake ninja-build git build-essential ccache

# Point ccache cache at Google Drive so it survives session resets
mkdir -p /drive/MyDrive/engram_cache/ccache
export CCACHE_DIR=/drive/MyDrive/engram_cache/ccache
# Google Drive (FUSE) doesn't support hard links — ccache needs a local temp dir
export CCACHE_TEMPDIR=/tmp
ccache --max-size=5G

echo '✅ Tools installed'
cmake --version | head -1
nvcc --version | grep release
ccache --version | head -1
ccache -s || true

## 3. Clone Engram + build (or restore from Drive cache)

The compiled binary is stored in your Google Drive keyed by git commit hash.  
**First run:** takes 25–40 min. **Every run after:** ~10 seconds.

In [ ]:
import subprocess, os, shutil, time

CACHE_DIR  = '/drive/MyDrive/engram_cache'
REPO_URL   = 'https://github.com/pilsy/engram.git'
REPO_DIR   = '/content/engram'
BINARY     = '/content/engram/build/engram'

# ── Clone / pull ──────────────────────────────────────────────────────────
if os.path.isdir(REPO_DIR):
    print('Repo exists — pulling latest...')
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    print(f'Cloning {REPO_URL}...')
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)

commit = subprocess.check_output(
    ['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD'], text=True).strip()
print(f'\nCommit: {commit}')
subprocess.run(['git', '-C', REPO_DIR, 'log', '--oneline', '-3'], check=True)

# ── Check Drive cache ─────────────────────────────────────────────────────
cached_binary = f'{CACHE_DIR}/engram-{commit}'

if os.path.isfile(cached_binary):
    print(f'\n✅ Cached binary found for {commit} — skipping build!')
    os.makedirs(f'{REPO_DIR}/build', exist_ok=True)
    shutil.copy2(cached_binary, BINARY)
    os.chmod(BINARY, 0o755)
    size = os.path.getsize(BINARY) / 1024 / 1024
    print(f'   Restored {size:.1f} MB in seconds.')
else:
    print(f'\n⚙️  No cache for {commit} — building from source.')
    print('   First build takes 25–40 min on Colab. Grab a coffee. ☕')
    print('   ccache will speed up future re-runs.')
    print('   Progress dots print every 30s...')

    env = os.environ.copy()
    env['CCACHE_DIR']     = f'{CACHE_DIR}/ccache'
    # Google Drive (FUSE) doesn't support hard links — keep ccache temp on local disk
    env['CCACHE_TEMPDIR'] = '/tmp'
    env['CC']  = 'ccache gcc'
    env['CXX'] = 'ccache g++'
    env['CUDACXX'] = '/usr/local/cuda/bin/nvcc'

    # Configure
    subprocess.run([
        'cmake', '-B', f'{REPO_DIR}/build',
        '-S', REPO_DIR,
        '-DCMAKE_BUILD_TYPE=Release',
        '-DGGML_CUDA=ON',
        '-DGGML_CUDA_FORCE_MMQ=OFF',
        '-DLLAMA_BUILD_SERVER=OFF',
        '-DLLAMA_BUILD_TESTS=OFF',
        '-DLLAMA_BUILD_EXAMPLES=OFF',
        '-GNinja',
        f'-DCMAKE_C_COMPILER_LAUNCHER=ccache',
        f'-DCMAKE_CXX_COMPILER_LAUNCHER=ccache',
    ], check=True, env=env, capture_output=True)
    print('✅ CMake configured')

    # Build with live dot progress
    t0 = time.time()
    build_log = []
    proc = subprocess.Popen(
        ['cmake', '--build', f'{REPO_DIR}/build', '-j2'],
        env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    last_dot = time.time()
    last_line = ''
    for line in proc.stdout:
        build_log.append(line)
        last_line = line.strip()
        # Print a dot every 30s so Colab doesn't think we're idle
        if time.time() - last_dot > 30:
            elapsed = int(time.time() - t0)
            print(f'  [{elapsed//60}m{elapsed%60:02d}s] building... ({last_line[:70]})', flush=True)
            last_dot = time.time()
    proc.wait()

    elapsed = int(time.time() - t0)
    if proc.returncode != 0:
        # Print last 60 lines so we can see what went wrong
        print('\n❌ Build failed! Last 60 lines of output:')
        print(''.join(build_log[-60:]))
        raise RuntimeError(f'Build failed after {elapsed//60}m{elapsed%60}s (see output above)')

    print(f'\n✅ Build complete in {elapsed//60}m{elapsed%60:02d}s')

    # Save to Drive cache
    shutil.copy2(BINARY, cached_binary)
    size = os.path.getsize(BINARY) / 1024 / 1024
    print(f'   Cached {size:.1f} MB to Drive as engram-{commit}')

    # Show ccache stats
    subprocess.run(['ccache', '-s'], env=env)

## 4. Download model

**Mistral 7B Instruct v0.2 Q4_K_M** (~4.1 GB) — also cached to Drive.

In [ ]:
import os, shutil

# ── Model config ───────────────────────────────────────────────────────────
MODEL_REPO = 'TheBloke/Mistral-7B-Instruct-v0.2-GGUF'
MODEL_FILE = 'mistral-7b-instruct-v0.2.Q4_K_M.gguf'
# ──────────────────────────────────────────────────────────────────────────

CACHE_DIR  = '/drive/MyDrive/engram_cache'
MODEL_PATH = f'/content/{MODEL_FILE}'
CACHED_MODEL = f'{CACHE_DIR}/{MODEL_FILE}'

if os.path.isfile(MODEL_PATH):
    print(f'✅ Model already at {MODEL_PATH}')
elif os.path.isfile(CACHED_MODEL):
    print(f'Restoring model from Drive cache...')
    shutil.copy2(CACHED_MODEL, MODEL_PATH)
    print(f'✅ Restored {os.path.getsize(MODEL_PATH)/1e9:.2f} GB')
else:
    print(f'Downloading {MODEL_FILE} from HuggingFace (~4.1 GB)...')
    url = f'https://huggingface.co/{MODEL_REPO}/resolve/main/{MODEL_FILE}'
    os.system(f'wget -q --show-progress -O "{MODEL_PATH}" "{url}"')
    # Save to Drive for next time
    print('Caching to Drive...')
    shutil.copy2(MODEL_PATH, CACHED_MODEL)
    print(f'✅ Downloaded and cached: {os.path.getsize(MODEL_PATH)/1e9:.2f} GB')

print(f'MODEL_PATH = {MODEL_PATH}')

## 5. Start the Engram server

In [ ]:
import subprocess, time, requests, os

MODEL_PATH  = f'/content/mistral-7b-instruct-v0.2.Q4_K_M.gguf'
ENGRAM_BIN  = '/content/engram/build/engram'
ENGRAM_HOST = 'http://127.0.0.1:8080'

# Kill any stale process
subprocess.run(['pkill', '-f', 'engram'], capture_output=True)
time.sleep(1)
os.makedirs('/content/sessions', exist_ok=True)

# Auto-detect VRAM and set GPU layers accordingly
try:
    vram_mb = int(subprocess.check_output(
        ['nvidia-smi', '--query-gpu=memory.total', '--format=csv,noheader,nounits'],
        text=True).strip().split('\n')[0])
    # Mistral 7B Q4_K_M: ~4.1 GB weights. T4 has 15 GB — use all 32 layers.
    GPU_LAYERS = 32 if vram_mb >= 8000 else 20 if vram_mb >= 5000 else 0
    print(f'GPU VRAM: {vram_mb} MB  →  GPU layers: {GPU_LAYERS}')
except Exception:
    GPU_LAYERS = 0
    print('No GPU → CPU-only')

log = open('/content/engram.log', 'w')
proc = subprocess.Popen([
    ENGRAM_BIN,
    '--host',         '127.0.0.1',
    '--port',         '8080',
    '--model',        MODEL_PATH,
    '--max-hot',      '2',
    '--max-warm',     '8',
    '--sessions-dir', '/content/sessions',
    '--threads',      '4',
    '--gpu-layers',   str(GPU_LAYERS),
], stdout=log, stderr=log)

print(f'Engram PID {proc.pid} — waiting for ready...')
for _ in range(60):
    time.sleep(2)
    try:
        r = requests.get(f'{ENGRAM_HOST}/sessions', timeout=2)
        if r.status_code == 200:
            print(f'\n✅ Server up: {r.json()}')
            break
    except Exception:
        print('.', end='', flush=True)
else:
    print('\n❌ Server failed to start — log:')
    print(open('/content/engram.log').read()[-3000:])

In [ ]:
!tail -50 /content/engram.log

## 6. Benchmark helpers

In [ ]:
import requests, time

ENGRAM_HOST = 'http://127.0.0.1:8080'
W = 74

PROMPTS = [
    'What is a KV cache in the context of transformer models?',
    'How does prefill differ from decode in LLM inference?',
    'Why does prefill cost scale quadratically with context length?',
    'What would happen if you could skip prefill on subsequent calls?',
    'How might you serialize a KV cache to disk and restore it later?',
    'What tradeoffs exist between VRAM, RAM, and NVMe for cache storage?',
    'How would you handle context window overflow in a persistent cache server?',
    'What would a benchmark harness for this look like?',
    'How does Flash Attention interact with a persistent KV cache design?',
    'What are the most impactful use cases for persistent KV caches in agentic workflows?',
]

def fmt(ms):
    if ms is None: return 'n/a'
    return f'{ms/1000:.2f}s' if ms >= 1000 else f'{ms:.0f}ms'

def bar(val, max_val, width=28):
    n = round((val / max_val) * width) if max_val > 0 else 0
    return '█' * n + '░' * (width - n)

def epost(path, body=None):
    r = requests.post(f'{ENGRAM_HOST}{path}', json=body, timeout=300)
    r.raise_for_status()
    return r.json()

def eget(path):
    r = requests.get(f'{ENGRAM_HOST}{path}', timeout=30)
    r.raise_for_status()
    return r.json()

def edel(path, body=None):
    r = requests.delete(f'{ENGRAM_HOST}{path}', json=body, timeout=30)
    r.raise_for_status()
    return r.json()

print('✅ Helpers ready')

## 7. Baseline benchmark

Each turn evicts the session and creates a fresh one — forcing a full re-prefill of the entire
conversation history. This is what a stateless inference server does.

In [ ]:
N_TURNS   = 10
N_PREDICT = 128
N_CTX     = 4096

print('╔══════════════════════════════════════════════╗')
print('║    baseline — stateless, full re-prefill     ║')
print('╚══════════════════════════════════════════════╝')
print(f'  turns={N_TURNS}  n_predict={N_PREDICT}  gpu_layers={GPU_LAYERS}')
print()
print('  Each turn: evict session → new session → prefill FULL history.')
print('  This is O(n²): prefill grows with every turn.')
print()

print('-' * W)
print(f" {'Turn':<5} {'Prompt tok':<14} {'Prefill':<12} {'Decode':<12} Slowdown")
print('-' * W)

messages = []
baseline_results = []
first_prefill = None

for i in range(N_TURNS):
    user_msg = PROMPTS[i % len(PROMPTS)]
    messages.append({'role': 'user', 'content': user_msg})

    sid = f'baseline-turn-{i}-{int(time.time())}'

    # Fresh session every turn = no cache
    epost('/session/create', {'session_id': sid, 'n_ctx': N_CTX, 'n_gpu_layers': GPU_LAYERS})

    r = epost('/session/chat', {
        'session_id': sid,
        'messages':   list(messages),
        'n_predict':  N_PREDICT,
        'temperature': 0.7,
        'top_p': 0.9,
        'top_k': 40,
    })

    edel('/session/evict', {'session_id': sid})

    if r.get('text'):
        messages.append({'role': 'assistant', 'content': r['text']})

    ms_pre = r.get('ms_prefill', 0)
    ms_dec = r.get('ms_decode', 0)
    n_tok  = r.get('n_tokens_prompt', 0)

    if first_prefill is None: first_prefill = ms_pre
    slowdown = f"{ms_pre/first_prefill:.1f}×" if first_prefill else '—'

    baseline_results.append({'turn': i+1, 'n_tok': n_tok, 'ms_pre': ms_pre,
                              'ms_dec': ms_dec, 'user': user_msg, 'reply': r.get('text','')})
    print(f" {i+1:<5} {n_tok:<14} {fmt(ms_pre):<12} {fmt(ms_dec):<12} {slowdown}")

print('-' * W)
print()

b1, bN = baseline_results[0], baseline_results[-1]
print('Baseline Summary')
print('─' * 20)
print(f"  Turn 1  : {fmt(b1['ms_pre'])}  ({b1['n_tok']} tokens)")
print(f"  Turn {N_TURNS}  : {fmt(bN['ms_pre'])}  ({bN['n_tok']} tokens)")
if b1['ms_pre']:
    print(f"  Slowdown: {bN['ms_pre']/b1['ms_pre']:.1f}×  ← this is what Engram eliminates")
print()
mp = max(r['ms_pre'] for r in baseline_results)
for r in baseline_results:
    print(f"  [{r['turn']:2d}] {bar(r['ms_pre'],mp)} {fmt(r['ms_pre'])}  ({r['n_tok']} tok)")

## 8. Engram chat benchmark

**Same prompts. Same full message history passed every turn.**  
One persistent session. Engram extracts the delta tokens and only prefills those.

In [ ]:
print('╔══════════════════════════════════════════════╗')
print('║       engram chat — delta-only prefill       ║')
print('╚══════════════════════════════════════════════╝')
print(f'  turns={N_TURNS}  n_predict={N_PREDICT}  gpu_layers={GPU_LAYERS}')
print()
print('  One persistent session. Delta tokens only hit prefill.')
print()

SID = f'engram-{int(time.time())}'
epost('/session/create', {'session_id': SID, 'n_ctx': N_CTX, 'n_gpu_layers': GPU_LAYERS})

print('-' * W)
print(f" {'Turn':<5} {'Cached':<10} {'Delta':<9} {'Prefill':<12} {'Decode':<12} {'Hit':<4} Speedup")
print('-' * W)

messages = []
engram_results = []
first_prefill = None

for i in range(N_TURNS):
    user_msg = PROMPTS[i % len(PROMPTS)]
    messages.append({'role': 'user', 'content': user_msg})

    r = epost('/session/chat', {
        'session_id':  SID,
        'messages':    list(messages),
        'n_predict':   N_PREDICT,
        'temperature': 0.7,
        'top_p': 0.9,
        'top_k': 40,
    })

    if r.get('text'):
        messages.append({'role': 'assistant', 'content': r['text']})

    ms_pre  = r.get('ms_prefill', 0)
    ms_dec  = r.get('ms_decode', 0)
    n_delta = r.get('n_tokens_prompt', 0)
    n_cache = r.get('n_tokens_in_cache', 0)
    hit     = r.get('cache_hit', False)

    if first_prefill is None: first_prefill = ms_pre
    speedup  = f"{first_prefill/ms_pre:.1f}×" if ms_pre else '—'
    hit_mark = '✓' if hit else '✗'

    engram_results.append({'turn': i+1, 'n_cache': n_cache, 'n_delta': n_delta,
                           'ms_pre': ms_pre, 'ms_dec': ms_dec, 'hit': hit,
                           'user': user_msg, 'reply': r.get('text','')})
    print(f" {i+1:<5} {n_cache:<10} {n_delta:<9} {fmt(ms_pre):<12} {fmt(ms_dec):<12} {hit_mark:<4} {speedup}")

print('-' * W)

status = eget(f'/session/status?session_id={SID}')
edel('/session/evict', {'session_id': SID})

print()
e1, eN = engram_results[0], engram_results[-1]
print('Engram Summary')
print('─' * 20)
print(f"  Turn 1  : {fmt(e1['ms_pre'])}  ({e1['n_delta']} delta tok, cold start)")
print(f"  Turn {N_TURNS}  : {fmt(eN['ms_pre'])}  ({eN['n_delta']} delta tok, cached={eN['n_cache']})")
if e1['ms_pre']:
    print(f"  Creep   : {eN['ms_pre']/e1['ms_pre']:.1f}×  (O(delta×cached) attention — expected)")
print(f"  Total cache: {status.get('n_tokens_used','?')} tokens")
print()
mp = max(r['ms_pre'] for r in engram_results)
for r in engram_results:
    print(f"  [{r['turn']:2d}] {bar(r['ms_pre'],mp)} {fmt(r['ms_pre'])}  (Δ={r['n_delta']} cached={r['n_cache']})")

## 9. Side-by-side comparison

In [ ]:
print('=' * W)
print('  SIDE-BY-SIDE: Baseline vs Engram Chat')
print('=' * W)
print(f" {'Turn':<5} {'Baseline prefill':<20} {'Engram prefill':<20} Speedup")
print('-' * W)

speedups = []
for b, e in zip(baseline_results, engram_results):
    sp = b['ms_pre'] / e['ms_pre'] if e['ms_pre'] else 0
    speedups.append(sp)
    print(f"  {b['turn']:<5} {fmt(b['ms_pre']):<20} {fmt(e['ms_pre']):<20} {sp:.1f}×")

print('=' * W)
print()

b1, bN = baseline_results[0], baseline_results[-1]
e1, eN = engram_results[0],   engram_results[-1]

baseline_slowdown = bN['ms_pre'] / b1['ms_pre'] if b1['ms_pre'] else 0
engram_creep      = eN['ms_pre'] / e1['ms_pre'] if e1['ms_pre'] else 0
final_speedup     = bN['ms_pre'] / eN['ms_pre'] if eN['ms_pre'] else 0

print(f"  Baseline   turn 1→{N_TURNS}: {fmt(b1['ms_pre'])} → {fmt(bN['ms_pre'])}  ({baseline_slowdown:.1f}× slowdown)")
print(f"  Engram     turn 1→{N_TURNS}: {fmt(e1['ms_pre'])} → {fmt(eN['ms_pre'])}  ({engram_creep:.1f}× creep)")
print()
print(f"  ⚡ Engram is {final_speedup:.0f}× faster on turn {N_TURNS} prefill")
print()
print(f"  The {baseline_slowdown:.0f}× baseline slowdown is eliminated.")
print(f"  The {engram_creep:.1f}× Engram creep is unavoidable physics:")
print(f"    {eN['n_delta']} new tokens × {eN['n_cache']} cached keys/values")
print(f"    formula: prefill ∝ delta_tokens × cached_tokens")
print()

# ASCII comparison chart
print('  Prefill per turn (same scale):')
print()
mp = max(r['ms_pre'] for r in baseline_results + engram_results)
print(f"  {'':4} {'── Baseline ──':^30} {'── Engram ──':^30}")
for b, e in zip(baseline_results, engram_results):
    bb = bar(b['ms_pre'], mp, 28)
    eb = bar(e['ms_pre'], mp, 28)
    print(f"  [{b['turn']:2d}] {bb} {eb}")
print(f"       {'0':>1}{'':>26} {fmt(mp)}")

## 10. Conversation transcript

In [ ]:
def wrap(text, width=64):
    lines = []
    for para in text.split('\n'):
        if len(para) <= width:
            lines.append(para); continue
        cur = ''
        for w in para.split():
            if not cur: cur = w
            elif len(cur)+1+len(w) <= width: cur += ' '+w
            else: lines.append(cur); cur = w
        if cur: lines.append(cur)
    return lines or ['']

print('━' * W)
print('  CONVERSATION TRANSCRIPT (Engram)')
print('━' * W)

for r in engram_results:
    hit = '✓ cache hit' if r['hit'] else '✗ cold'
    print(f"\n  ┌─ Turn {r['turn']}  prefill={fmt(r['ms_pre'])}  decode={fmt(r['ms_dec'])}  {hit}")
    print(f"  │  Δ={r['n_delta']} tok  cached={r['n_cache']} tok")
    print(f"  │\n  │  👤 User:")
    for l in wrap(r['user']): print(f'  │     {l}')
    print(f"  │\n  │  🤖 Engram:")
    for l in wrap((r['reply'] or '(no response)').strip()): print(f'  │     {l}')
    print(f"  └" + '─'*(W-4))

print('\n' + '━' * W)

## 11. Teardown

In [ ]:
import subprocess
subprocess.run(['pkill', '-f', 'engram'], capture_output=True)
!rm -rf /content/sessions
print('✅ Done')